# 基于开源仓的算子开发
在熟悉开源仓的算子整体结构后，本节将重点讲解算子的开发流程，以及如何将自研的单算子工程合规贡献至开源仓库，核心内容如下：
- 环境准备
- 开源仓算子开发总览          
- 开源仓算子代码编写   
- 开源仓算子编译部署   
- 开源仓算子验证
- 单算子工程迁移贡献

---

## 1. 环境准备
本文所有内容均存放于Sources文件夹。  
在开始创建算子工程前，先要对jupyter环境进行初始化。以下代码完成了初始化并将环境中的变量导入jupyter环境，并完成代码目录的创建。

In [ ]:
!mkdir -p Sources/06.03

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("\n🎉 Environment initialization process completed successfully!")

ops-math仓创建新算子工程时依赖gawk，执行以下命令安装gawk

In [ ]:
%%bash
set -e

GAWK_VERSION="5.3.0"
INSTALL_DIR="$HOME/.local"
RC_FILE="$HOME/.bashrc"  # 固定使用.bashrc，不存在则创建
PATH_LINE="export PATH=$INSTALL_DIR/bin:\$PATH"
# 确保.bashrc存在，且只添加一次PATH
touch "$RC_FILE" 
source "$RC_FILE"
if command -v gawk &> /dev/null; then
    echo "✅ gawk 已存在，版本：$(gawk --version | head -n1)"
    echo "✅ 跳过编译安装"
    exit 0
fi

[ ! -f "gawk-$GAWK_VERSION.tar.gz" ] && wget https://ftp.gnu.org/gnu/gawk/gawk-$GAWK_VERSION.tar.gz

tar -zxf gawk-$GAWK_VERSION.tar.gz --overwrite 
cd gawk-$GAWK_VERSION
./configure --prefix=$INSTALL_DIR
make -j$(nproc)
make install

grep -q "$INSTALL_DIR/bin" "$RC_FILE" || echo "$PATH_LINE" >> "$RC_FILE"
source "$RC_FILE"

echo "gawk 安装完成，路径：$(which gawk)"

In [ ]:
import os
os.environ["PATH"] = f"{os.environ['HOME']}/.local/bin:{os.environ['PATH']}"

---

## 2. 开源仓算子开发总览
开源算子仓库提供标准化的算子开发流程，开发者可遵循该流程完成算子开发，并将成果合规贡献至开源仓。

<img src="./images/opensource_repo_operator_dev_flow.png" alt="开源仓算子开发流程" width="700px">

**环境准备说明**：
- CANN软件安装，需要按照[06.01_chapter_intro](./06.01_chapter_intro.ipynb)章节完成准备，在线体验环境无需手动安装。
- 开源源码仓下载：本章将以 ops-math 仓库为实操案例展开说明，可执行以下命令完成该仓库的下载。

In [ ]:
!cd Sources/06.03;git clone https://gitcode.com/cann/ops-math.git

---

## 3. 开源仓算子代码编写
开源仓库拉取完成后，即可遵循开源仓内置的算子标准化开发流程，开展算子的全流程开发工作。

### 3.1 算子分析
首先需要明确算子的计算逻辑、数据需求及所需调用的Ascend C编程接口。下文以Add算子为例，展示这一分析过程。

- 1.明确算子的数学表达式及计算逻辑。

    Add算子的数学表达式为：

    $\vec{z}$ =$\vec{x}$ +$\vec{y}$

    计算逻辑是数据需经历“搬运-计算-搬运”三个阶段：
    - 将输入数据从Global Memory搬运至Local Memory。
    - 在Local Memory中使用Ascend C计算接口执行矢量加法计算。
    - 将计算结果从Local Memory搬运回Global Memory。

- 2.明确输入和输出。
    - 输入数量：2个，分别为 x 和 y。
    - 输出数量：1个，为 z。
    - 数据类型：输入与输出均为 float 类型。
    - 数据形状（Shape）： 输入Shape为 (8, 2048)，输出Shape与输入相同。
    - 数据布局（Format）：ND。

- 3.确定核函数名称和参数。

    根据输入输出分析，确定核函数参数，定义核函数原型：
    - 核函数名称：add_custom
    - 参数列表：
    - x, y: 输入数据在Global Memory上的起始地址。
    - z: 输出数据在Global Memory上的起始地址。

- 4.确定算子实现所需接口。

    基于实现流程，确定需要调用的关键API类别：
    - 数据搬运：使用 DataCopy 接口完成Global与Local Memory间的数据搬移。
    - 矢量计算：使用矢量 Add 接口实现 x 与 y 的逐元素相加。
    - 内存管理：使用 AllocTensor 与 FreeTensor 为LocalTensor申请和释放内存。
    - 任务同步：使用 EnQue、DeQue 接口实现流水任务间的数据通信与同步。

通过以上分析，得到Ascend C Add算子的设计规格如下：

<table style="float: left; border-collapse: collapse; margin: 0 10px 10px 0; font-size: 14px;">
  <tr style="background-color:#f0f0f0">
    <td align="center"><strong>算子类型（OpType）</strong></td>
    <td align="center" colspan="4"><strong>Add</strong></td>
  </tr>
  <tr>
    <td align="center" rowspan="3"><strong>算子输入</strong></td>
    <td align="center"><strong>name</strong></td>
    <td align="center"><strong>shape</strong></td>
    <td align="center"><strong>data type</strong></td>
    <td align="center"><strong>format</strong></td>
  </tr>
  <tr>
    <td align="left">x</td>
    <td align="left">(8, 2048)</td>
    <td align="left">float、int32</td>
    <td align="left">ND</td>
  </tr>
  <tr>
    <td align="left">y</td>
    <td align="left">(8, 2048)</td>
    <td align="left">float、int32</td>
    <td align="left">ND</td>
  </tr>
  <tr>
    <td align="center"><strong>算子输出</strong></td>
    <td align="left">z</td>
    <td align="left">(8, 2048)</td>
    <td align="left">float、int32</td>
    <td align="left">ND</td>
  </tr>
  <tr>
    <td align="center"><strong>支持设备</strong></td>
    <td align="center" colspan="4">Ascend910B</td>
  </tr>
  <tr>
    <td align="center"><strong>核函数名</strong></td>
    <td align="center" colspan="4">add_custom</td>
  </tr>
</table>
<div style="clear: both;"></div>

### 3.2 创建算子工程
开源仓提供的 build.sh 脚本内置了算子工程生成参数 --genops，可在指定目录下快速生成空的算子工程模板。创建命令如下：

```shell
bash build.sh --genop=${op_class}/${op_name}
```

**参数说明：**

- `${op_class}`：算子类型目录，若为外部贡献的算子，需统一放置在 experimental/math 目录下；若指定目录不存在，脚本会自动创建该目录，**该参数为必填项，不支持省略**。
- `${op_name}`：算子名称，需采用**小写 + 下划线**的命名规范（snake_case），例如 AddExample 算子对应的名称为 add_example。

**实操示例：**

开源仓中用户贡献的算子均需归属到 experimental 目录下，以创建 experimental/math 目录下的 add_custom 算子为例，可执行以下命令：

In [ ]:
%%bash
cd Sources/06.03/ops-math 
# 预防重复生成，先删除add_custom目录
rm -rf experimental/math/add_custom
bash build.sh --genop=experimental/math/add_custom

可执行以下命令，查看生成的算子工程目录结构：

In [ ]:
!ls -l ./Sources/06.03/ops-math/experimental/math/add_custom

### 3.3 算子定义
算子定义文件命名格式为 `${op_name}`_def.cpp，该文件在自定义算子工程生成后会自动生成于 op_host 目录下。作为算子的核心信息配置文件，需根据算子的实际功能与参数规格完成适配修改。

**实操示例：**

本次实验中，生成的算子定义文件为experimental/math/add/op_host/add_custom_def.cpp，可执行以下命令查看生成后的文件。

In [ ]:
!cat ./Sources/06.03/ops-math/experimental/math/add_custom/op_host/add_custom_def.cpp

基于上文的算子分析，执行以下代码段，按实际算子的功能与参数规格修改算子定义文件：

In [ ]:
%%writefile ./Sources/06.03/ops-math/experimental/math/add_custom/op_host/add_custom_def.cpp

#include "register/op_def_registry.h"

namespace ops {
class AddCustom : public OpDef {
public:
    explicit AddCustom(const char* name) : OpDef(name)
    {
        // 输入参数说明
        this->Input("x1")                                       // 输入x1定义
            .ParamType(REQUIRED)                                // 必选输入
            .DataType({ge::DT_FLOAT, ge::DT_INT32})             // 支持数据类型
            .Format({ge::FORMAT_ND, ge::FORMAT_ND})             // 支持format格式
            .UnknownShapeFormat({ge::FORMAT_ND, ge::FORMAT_ND}) // 未确定大小shape对应format格式
            .AutoContiguous();                                  // 内存自动连续化
        this->Input("x2")                                       // 输入x2定义
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT, ge::DT_INT32})
            .Format({ge::FORMAT_ND, ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND, ge::FORMAT_ND})
            .AutoContiguous();
        this->Output("y") // 输出y定义
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT, ge::DT_INT32})
            .Format({ge::FORMAT_ND, ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND, ge::FORMAT_ND})
            .AutoContiguous();

        this->AICore().AddConfig("ascend910b");
    }
};
OP_ADD(AddCustom); // 添加算子信息库
} // namespace ops

可以看到，本原型文件的内容与[03.02_operator_engineering_intro.ipynb](../03_intermediate_vector_operator_development/03.02_operator_engineering_intro.ipynb)中自定义算子工程的原型定义完全一致。核心区别仅在于，**开源仓将该原型文件单独拆分出来，以此保证编程逻辑的独立性**。

### 3.4 tiling实现
Tiling 实现需交付三个核心文件：`${op_name}`_tiling.cpp、`${op_name}`_tiling_key.h、`${op_name}`_tiling_data.h。关于 TilingKey 的核心概念已在前序章节介绍，此处做简要回顾：

TilingKey 是算子内部用于区分不同 Kernel 实现的核心机制，其设计思路类似 C++ 的 Template 模板机制，可有效减少不必要的指令缓存缺失（icache miss）和标量计算耗时，从而优化单次 Kernel 调用的性能。TilingKey 最基础的使用逻辑如下：

<img src="./images/tilingkey_call_flow.png" alt="tilingkey调用流程" width="700px">

由于 TilingKey 通常表现为无明确语义的长数字串，易记易理解性较差，因此开源仓专门设计了 Tiling 模板编程方法来简化使用，流程如下：

<img src="./images/opensource_repo_tilingkey_call_flow.png" alt="开源仓tilingkey调用流程" width="700px">

其中，`${op_name}`_tiling_key.h 用于定义 TilingKey 模板参数，而 TilingKey 的生成逻辑则在 `${op_name}`_tiling.cpp 中实现。

#### 3.4.1 ${op_name}_tiling_key.h
该文件在自定义算子工程生成后，自动生成于 op_kernel 目录下，核心作用是定义 Tiling 模板的参数规格。

**实操示例：**

本次实验中，生成的 add_custom_tiling_key.h 文件路径为 experimental/math/add_custom/op_kernel/add_custom_tiling_key.h，可执行以下命令查看该文件的初始内容：

In [ ]:
!cat ./Sources/06.03/ops-math/experimental/math/add_custom/op_kernel/add_custom_tiling_key.h

该文件定义了模板参数 schMode，其取值范围限定为 0 或 1；当前仅配置了一组模板参数（即 schMode 可在 0、1 中任选其一）。

鉴于该算子无需配置多组 TilingKey，可对本文件做简化处理：仅保留 schMode 取值为 0 的配置即可。请执行以下代码完成文件修改。

In [ ]:
%%writefile Sources/06.03/ops-math/experimental/math/add_custom/op_kernel/add_custom_tiling_key.h

#ifndef __ADD_CUSTOM_TILING_KEY_H__
#define __ADD_CUSTOM_TILING_KEY_H__

#include "ascendc/host_api/tiling/template_argument.h"

#define ELEMENTWISE_TPL_SCH_MODE_0 0

ASCENDC_TPL_ARGS_DECL(
    AddCustom,
    ASCENDC_TPL_UINT_DECL(schMode, 1, ASCENDC_TPL_UI_LIST, ELEMENTWISE_TPL_SCH_MODE_0));

ASCENDC_TPL_SEL(ASCENDC_TPL_ARGS_SEL(
    ASCENDC_TPL_UINT_SEL(schMode, ASCENDC_TPL_UI_LIST, ELEMENTWISE_TPL_SCH_MODE_0)));

#endif

复杂一点的Tiling模板参数定义，如下所示：

```cpp
#include "ascendc/host_api/tiling/template_argument.h"

// 模板参数
ASCENDC_TPL_ARGS_DECL(AddCustomTemplate, // 算子OpType
ASCENDC_TPL_DATATYPE_DECL(D_T_X, C_DT_FLOAT, C_DT_FLOAT16, ASCENDC_TPL_INPUT(0)),  // DataType类型的模板参数定义：输入参数x的数据类型，取值范围为float16/float32, ASCENDC_TPL_INPUT(0)说明对应Kernel侧第0个输入
ASCENDC_TPL_DATATYPE_DECL(D_T_Y, C_DT_FLOAT, C_DT_FLOAT16, ASCENDC_TPL_INPUT(1)),  // DataType类型的模板参数定义：输入参数y的数据类型，取值范围为float16/float32, ASCENDC_TPL_INPUT(1)说明对应Kernel侧第1个输入
ASCENDC_TPL_DATATYPE_DECL(D_T_Z, C_DT_FLOAT, C_DT_FLOAT16, ASCENDC_TPL_OUTPUT(0)), // DataType类型的模板参数定义：输入参数z的数据类型，取值范围为float16/float32, ASCENDC_TPL_OUTPUT(0)说明对应Kernel侧第0个输出
ASCENDC_TPL_UINT_DECL(TILE_NUM, ASCENDC_TPL_8_BW, ASCENDC_TPL_UI_MIX, 2, 0, 2, 3, 5, 10, 12, 13, 9, 8),// 自定义UINT类型（无符号整形）的模板参数定义：模板参数为切分的块数，编码位宽为ASCENDC_TPL_8_BW即8比特，表示该模板参数的个数不超过8比特能表达的范围；ASCENDC_TPL_UI_MIX表示通过混合模式表达取值范围，有2组的数据{0-2}、{3-5}和穷举值10、12、13、9、8，最后结果为{0, 1, 2, 3, 4, 5, 10, 12, 13, 9, 8}
ASCENDC_TPL_BOOL_DECL(IS_SPLIT, 0, 1), // 自定义bool类型的模板参数定义：模板参数为是否切分标志位，取值范围为0和1，1表示切分，0表示不切分
);

// 模板参数组合
// 用于调用GET_TPL_TILING_KEY获取TilingKey时，接口内部校验TilingKey是否合法
ASCENDC_TPL_SEL(
    ASCENDC_TPL_ARGS_SEL(
    ASCENDC_TPL_KERNEL_TYPE_SEL(ASCENDC_TPL_AIV_ONLY), // Kernel类型选择，无需在模板参数声明中定义，在SEL中直接配置，所有ASCENDC_TPL_ARGS_SEL是否配置需要保持统一，如不配置将走自动推导流程
    ASCENDC_TPL_DATATYPE_SEL(D_T_X, C_DT_FLOAT16),
    ASCENDC_TPL_DATATYPE_SEL(D_T_Y, C_DT_FLOAT16),
    ASCENDC_TPL_DATATYPE_SEL(D_T_Z, C_DT_FLOAT16),
    ASCENDC_TPL_UINT_SEL(TILE_NUM, ASCENDC_TPL_UI_LIST, 1, 8),
    ASCENDC_TPL_BOOL_SEL(IS_SPLIT, 0, 1)
    ),
    ASCENDC_TPL_ARGS_SEL(
    ASCENDC_TPL_KERNEL_TYPE_SEL(ASCENDC_TPL_AIV_ONLY),
    ASCENDC_TPL_DATATYPE_SEL(D_T_X, C_DT_FLOAT),
    ASCENDC_TPL_DATATYPE_SEL(D_T_Y, C_DT_FLOAT),
    ASCENDC_TPL_DATATYPE_SEL(D_T_Z, C_DT_FLOAT),
    ASCENDC_TPL_UINT_SEL(TILE_NUM, ASCENDC_TPL_UI_LIST, 1, 8),
    ASCENDC_TPL_BOOL_SEL(IS_SPLIT, 0, 1)
    ),
);
```

其中，部分接口功能如下：

- **ASCENDC_TPL_ARGS_DECL(args0, ...)**：用于定义算子的模板参数。args0表示算子Optype，args1-argsn为后续为若干个DTYPE、FORMAT、UINT、BOOL的模板参数定义，分别通过ASCENDC_TPL_DTYPE_DECL、ASCENDC_TPL_FORMAT_DECL、ASCENDC_TPL_UINT_DECL、ASCENDC_TPL_BOOL_DECL进行定义。

- **ASCENDC_TPL_UINT_DECL(args0, args1, args2, ...)**：用于自定义UINT类型（无符号整形）的模板参数定义。args0为参数名，args1为最大位宽，模板参数的格式不能超过最大位宽。args2为参数定义的模式，支持ASCENDC_TPL_UI_RANGE(范围模式)/ASCENDC_TPL_UI_LIST(穷举模式)/ASCENDC_TPL_UI_MIX(混合模式)

- **ASCENDC_TPL_SEL(...)**：用于算子的模板参数整体组合。使用多个ASCENDC_TPL_ARGS_DSEL进行模板参数组合。

- **ASCENDC_TPL_ARGS_SEL(...)**：用于一个算子的模板参数组合，与前面模板参数定义匹配，使用SEL接口。

- **ASCENDC_TPL_UINT_SEL(args0, args1, args2, ...)**：UINT类型的模板参数组合。args0为参数名，args1为参数定义的模式，支持ASCENDC_TPL_UI_RANGE(范围模式)/ASCENDC_TPL_UI_LIST(穷举模式)/ASCENDC_TPL_UI_MIX(混合模式)，args2-argsn为后续若干个参数为ASCENDC_TPL_UINT_DECL中定义的参数范围子集

#### 3.4.2 ${op_name}_tiling_data.h
该文件在自定义算子工程生成后自动创建于 op_kernel 目录下，作为 Tiling 结构体的核心定义文件，需根据算子的实际 Tiling 策略完成适配修改。

在 Ascend C 开发体系中，Tiling 策略的核心载体是 C 语言结构体（struct），简称 Tiling 结构体，其关键规则如下：

- Tiling 结构体统一定义在专属头文件中（命名格式为 xxxx_custom_tiling.h），结构体中的每个参数对应输入数据的切分规则，同时决定计算过程的核心细节（如数据分块大小、并行粒度等）。

- 结构体的定义通常存放于 op_kernel 目录下的头文件中，在 Tiling 实现文件中完成实例化后，通过指针传递至 Kernel 函数中调用。

- 基于开源算子仓开发时，直接采用标准 C 语言 struct 语法定义即可，也可在定义阶段为结构体参数赋予初始值。

**实操示例：**

本次实验中，生成的 Tiling 结构体文件路径为 experimental/math/add_custom/op_kernel/add_custom_tiling_data.h，可执行以下命令查看该文件的初始内容：

In [ ]:
!cat ./Sources/06.03/ops-math/experimental/math/add_custom/op_kernel/add_custom_tiling_data.h

鉴于 add 算子的逻辑较为简单，仅需这两个 Tiling 结构体参数即可满足需求，因此本文件无需进行任何修改；也可执行以下代码块完成文件的覆盖重写：

In [ ]:
%%writefile Sources/06.03/ops-math/experimental/math/add_custom/op_kernel/add_custom_tiling_data.h

#ifndef __ADD_CUSTOM_TILLING_DATA_H__
#define __ADD_CUSTOM_TILLING_DATA_H__

struct AddCustomTilingData {
    int64_t totalLength;
    int64_t tileNum;
};
#endif

#### 3.4.3 ${op_name}_tiling.cpp
完成 Tiling 结构体定义后，需实现 Tiling 函数，该函数会基于算子输入输出的 Shape 等核心信息，计算并推导 Tiling 策略参数，将结果填充至 Tiling 结构体中，最终下发至 Kernel 侧执行。

Tiling 函数的入参和出参均为 TilingContext 对象，Tiling 结构体即存储于该对象内；运行时环境会自动完成 TilingContext 对象从 Host 侧到 Kernel 侧的跨侧传递，无需手动处理。

**实操示例：**

该 Tiling 函数的实现文件在自定义算子工程生成后，自动创建于 op_host 目录下。本次实验中，生成的 Tiling 实现文件路径为 experimental/math/add_custom/op_host/add_custom_tiling.cpp，可执行以下命令查看该文件的初始内容：

In [ ]:
!cat ./Sources/06.03/ops-math/experimental/math/add_custom/op_host/add_custom_tiling.cpp

接下来开始编写 Tiling 文件。该算子已在[03.02_operator_engineering_intro.ipynb](../03_intermediate_vector_operator_development/03.02_operator_engineering_intro.ipynb)中学习过，结合[03.05_tiling_template_attr_tbuf_workspace.ipynb](../03_intermediate_vector_operator_development/03.05_tiling_template_attr_tbuf_workspace.ipynb)的tiling模板化编程说明，此处不逐行解释代码含义。

因开源算子仓要求**必须编写TilingKey**，故本算子的Tiling代码如下：

In [ ]:
%%writefile Sources/06.03/ops-math/experimental/math/add_custom/op_host/add_custom_tiling.cpp

#include "log/log.h"
#include "util/math_util.h"
#include "op_host/tiling_util.h"
#include "op_host/tiling_templates_registry.h"
#include "../op_kernel/add_custom_tiling_data.h"
#include "../op_kernel/add_custom_tiling_key.h"

namespace optiling {

static ge::graphStatus TilingFunc(gert::TilingContext *context)
{
    uint32_t totalLength = context->GetInputShape(0)->GetOriginShape().GetShapeSize();
    context->SetBlockDim(8);
    AddCustomTilingData *tiling = context->GetTilingData<AddCustomTilingData>();
    tiling->totalLength = totalLength;
    tiling->tileNum = 1;
    uint64_t tilingKey = GET_TPL_TILING_KEY(ELEMENTWISE_TPL_SCH_MODE_0);
    context->SetTilingKey(tilingKey);

    return ge::GRAPH_SUCCESS;
}

IMPL_OP_OPTILING(AddCustom).Tiling(TilingFunc);
}  // namespace optiling

对比自定义算子后可发现以下核心差异点：

- **头文件引入差异**：本文件引入的头文件不同，但本质上无区别。gen_op 生成的头文件是开源仓库封装后的版本，而本文件使用的是算子工程生成、随 CANN 包安装完成即存在的原生头文件，二者功能完全等价，可任选其一使用。

- **Tiling 编程规范差异**：算子工程本身未强制约束 TilingKey 的编写方式，但[03.05_tiling_template_attr_tbuf_workspace.ipynb](../03_intermediate_vector_operator_development/03.05_tiling_template_attr_tbuf_workspace.ipynb)章节已介绍 Tiling 模板化编程方法；而开源算子仓对此有明确规范要求：**必须采用 Tiling 模板化编程**。因此若算子开发的最终目标是贡献至开源仓，需严格遵循该规范编写 TilingKey。

- **函数结构与宏定义差异**：本文件仅包含 TilingFunc 函数，并新增了 IMPL_OP_OPTILING 宏定义，这是开源仓算子与自定义算子工程最核心的区别。开源仓规范将 TilingFunc 单独拆分，保证编程逻辑的独立性；由于该函数独立存放于单个文件中，需通过 IMPL_OP_OPTILING 宏作为入口完成调用，反观 gen_op 工具生成的代码，除 TilingFunc 外还包含 TilingParse 函数：该函数用于执行 Tiling 前置处理操作，在简单业务场景下，可直接返回 ge::GRAPH_SUCCESS 表示处理完成，也可直接省略该函数不编写。

### 3.5 kernel实现
Kernel 实现需交付两个核心文件：\${op_name}.cpp（Kernel 入口文件）和 \${op_name}.h（Kernel 实现文件）。**也可和自定义算子保持一致，均写在${op_name}.cpp文件中**。

#### 3.5.1 ${op_name}.cpp
该文件是 Kernel 入口文件，包含主函数和调度逻辑。也就是核函数。由于 Kernel 侧需要接收 Host 侧下发的 Tiling 信息，核函数需按如下格式定义：

```cpp
__global__ __aicore__ void add_custom(GM_ADDR x, GM_ADDR y, GM_ADDR z, GM_ADDR workspace, GM_ADDR tiling)
```

> ⚠️ 重要规范：参数需严格按照「输入 → 输出 → workspace → tiling」的顺序排布，开发者请勿调整参数顺序。

**实操示例**

该文件在自定义算子工程生成后，自动创建于 op_kernel 目录下。本次实验中，生成的 Kernel 入口文件路径为 experimental/math/add_custom/op_kernel/add_custom.cpp，可执行以下命令查看该文件的初始内容：

In [ ]:
!cat ./Sources/06.03/ops-math/experimental/math/add_custom/op_kernel/add_custom.cpp

接下来开始编写核函数，由于引入了tiling模板化编程，所以代码如下：

In [ ]:
%%writefile Sources/06.03/ops-math/experimental/math/add_custom/op_kernel/add_custom.cpp

#include "add_custom.h"

enum class AddTilingKey : uint32_t
{
    TILING_KEY_EXAMPLE = 0,
};

template <uint32_t schMode>
 __global__ __aicore__ void add_custom(GM_ADDR x, GM_ADDR y, GM_ADDR z, GM_ADDR workspace, GM_ADDR tiling)
{
    REGISTER_TILING_DEFAULT(AddCustomTilingData);
    GET_TILING_DATA_WITH_STRUCT(AddCustomTilingData, tiling_data, tiling);
    if constexpr (schMode == static_cast<uint32_t>(AddTilingKey::TILING_KEY_EXAMPLE)) {
        KernelAdd<DTYPE_X1, DTYPE_X2, DTYPE_Y> op;
        op.Init(x, y, z, tiling_data.totalLength, tiling_data.tileNum);
        op.Process();
    }
}

以上为核函数的标准编写范式：尽管 schMode 仅支持取值 0，此处的条件判断看似无实际意义，但该写法为算子预留了可扩展空间，能够支撑算子工程的持续迭代升级，以及功能支持范围的持续拓展。

对比自定义算子工程的核函数，可发现以下核心差异：
- **仅保留核函数**：这是自定义算子工程与开源算子仓工程最核心的差异，开源算子仓为保障编程逻辑的独立性，将算子业务实现与核函数拆分为两个独立文件。

#### 3.5.2 ${op_name}.h
该文件是Kernel的实现文件，包含函数声明、结构定义、逻辑实现等。

**实操示例**

该文件在自定义算子工程生成后，自动创建于 op_kernel 目录下。本次实验中，生成的 Kernel 实现文件路径为 experimental/math/add_custom/op_kernel/add_custom.h，可执行以下命令查看该文件的初始内容：

In [ ]:
!cat ./Sources/06.03/ops-math/experimental/math/add_custom/op_kernel/add_custom.h

这里的核心代码与自定义算子中的并无区别，代码如下：

In [ ]:
%%writefile ./Sources/06.03/ops-math/experimental/math/add_custom/op_kernel/add_custom.h

#include "kernel_operator.h"
#include "kernel_tiling/kernel_tiling.h"
#include "add_custom_tiling_data.h"
#include "add_custom_tiling_key.h"

constexpr int32_t BUFFER_NUM = 1;  // tensor num for each queue

template <class dtypeX, class dtypeY, class dtypeZ>
class KernelAdd {
public:
    __aicore__ inline KernelAdd() {}
    __aicore__ inline void Init(GM_ADDR x, GM_ADDR y, GM_ADDR z, uint32_t totalLength, uint32_t tileNum)
    {
        this->blockLength = totalLength / AscendC::GetBlockNum();
        this->tileNum = tileNum;
        this->tileLength = this->blockLength / tileNum / BUFFER_NUM;
        xGm.SetGlobalBuffer((__gm__ dtypeX *)x + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
        yGm.SetGlobalBuffer((__gm__ dtypeY *)y + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
        zGm.SetGlobalBuffer((__gm__ dtypeZ *)z + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
        pipe.InitBuffer(inQueueX, BUFFER_NUM, this->tileLength * sizeof(dtypeX));
        pipe.InitBuffer(inQueueY, BUFFER_NUM, this->tileLength * sizeof(dtypeY));
        pipe.InitBuffer(outQueueZ, BUFFER_NUM, this->tileLength * sizeof(dtypeZ));
    }

    __aicore__ inline void Process()
    {
        int32_t loopCount = this->tileNum * BUFFER_NUM;
        for (int32_t i = 0; i < loopCount; i++) {
            CopyIn(i);
            Compute(i);
            CopyOut(i);
        }
        AscendC::printf("Core %d executed %d times in total\n",  AscendC::GetBlockIdx(), loopCount);
    }

private:
    __aicore__ inline void CopyIn(int32_t progress)
    {
        AscendC::LocalTensor<dtypeX> xLocal = inQueueX.AllocTensor<dtypeX>();
        AscendC::LocalTensor<dtypeY> yLocal = inQueueY.AllocTensor<dtypeY>();
        AscendC::DataCopy(xLocal, xGm[progress * this->tileLength], this->tileLength);
        AscendC::DataCopy(yLocal, yGm[progress * this->tileLength], this->tileLength);
        inQueueX.EnQue(xLocal);
        inQueueY.EnQue(yLocal);
    }
    __aicore__ inline void Compute(int32_t progress)
    {
        AscendC::LocalTensor<dtypeX> xLocal = inQueueX.DeQue<dtypeX>();
        AscendC::LocalTensor<dtypeY> yLocal = inQueueY.DeQue<dtypeY>();
        AscendC::LocalTensor<dtypeZ> zLocal = outQueueZ.AllocTensor<dtypeZ>();
        AscendC::Add(zLocal, xLocal, yLocal, this->tileLength);
        outQueueZ.EnQue<dtypeZ>(zLocal);
        inQueueX.FreeTensor(xLocal);
        inQueueY.FreeTensor(yLocal);
    }
    __aicore__ inline void CopyOut(int32_t progress)
    {
        AscendC::LocalTensor<dtypeZ> zLocal = outQueueZ.DeQue<dtypeZ>();
        AscendC::DataCopy(zGm[progress * this->tileLength], zLocal, this->tileLength);
        outQueueZ.FreeTensor(zLocal);
    }

private:
    AscendC::TPipe pipe;
    AscendC::TQue<AscendC::TPosition::VECIN, BUFFER_NUM> inQueueX;
    AscendC::TQue<AscendC::TPosition::VECIN, BUFFER_NUM> inQueueY;
    AscendC::TQue<AscendC::TPosition::VECOUT, BUFFER_NUM> outQueueZ;
    AscendC::GlobalTensor<dtypeX> xGm;
    AscendC::GlobalTensor<dtypeY> yGm;
    AscendC::GlobalTensor<dtypeZ> zGm;
    uint32_t blockLength;
    uint32_t tileNum;
    uint32_t tileLength;
};

对比自定义算子工程，可发现以下核心差异点：
- **仅保留算子核心业务逻辑**：这是自定义算子工程与开源算子仓工程最核心的差异，开源算子仓为保障编程逻辑的独立性，将算子业务实现与核函数拆分为两个独立文件。

至此，整个算子代码编写已经完成。

---

## 4.开源仓算子编译部署
本步骤主要分为以下三个核心环节：
- **aclnn适配**：算子完成开发与编译后，通常会自动生成 aclnn 接口（一套基于 C 语言的标准化 API），开发者可直接在应用程序中调用该接口实现算子调用。开源算子仓已对此环节做了优化，在非特殊编译参数的场景下，无需额外关注该适配环节，本节不做展开说明。
- **算子入图适配**：自定义算子工程里有 Shape/Dtype 推导函数，但前文代码编写没讲，这是开源仓分离设计的核心原因：仅用 aclnn 调用单算子（不入图），就不用写这些推导函数。
- **编译部署**：将开发完成的算子编译为 run 包，并部署至目标运行环境中。

### 4.1 算子入图适配（可选）
在深度学习中，当一个算子被加入计算图时，为确保图的正确性和后续的编译、优化、执行流程顺利进行，通常需要为该算子实现两个关键的推导函数：
- InferShape：用于推导输出张量的形状（shape）。
- InferDataType：用于推导输出张量的数据类型（dataType）。

这样可以在图执行之前，知道各Tensor的数据类型和形状，提前校验其正确性；

同时提前推理出算子的输出张量描述，包括张量的形状、数据类型及数据排布格式等信息，算子构图准备阶段就可以为所有的张量静态分配内存，避免动态内存分配带来的开销。

<img src="./images/operator_graph_adaptation_schematic.png" alt="算子入图适配示意图" width="350px">

在开源算子仓的开发规范中，算子入图相关交付件的编写规则如下：
- **若算子无需接入计算图（入图）：无需编写入图相关交付件（自动生成的 infershape 文件可直接删除）**；
- 若算子需要入图：需补充以下交付件：
```shell
${op_name}                              # 替换为实际算子名的小写下划线形式
├── op_host                             # Host侧实现
│   └── ${op_name}_infershape.cpp       # InferShape实现，实现算子形状推导，在运行时推导输出shape
├── op_graph                            # 图融合相关实现
│   ├── CMakeLists.txt                  # op_graph侧cmakelist文件
│   ├── ${op_name}_graph_infer.cpp      # InferDataType文件，实现算子类型推导，在运行时推导输出dataType
└── └── ${op_name}_proto.h              # 算子原型定义，用于图优化和融合阶段识别算子
```

#### 4.1.1 ${op_name}_infershape.cpp
InferShape实现，实现算子形状推导，在运行时推导输出shape。

**实操示例：**

本次实验中，生成的 infershape 实现文件路径为 experimental/math/add_custom/op_host/add_custom_infershape.cpp，可执行以下命令查看该文件的初始内容：

In [ ]:
!cat ./Sources/06.03/ops-math/experimental/math/add_custom/op_host/add_custom_infershape.cpp

该文件的初始实现逻辑为：直接将算子输出张量的 Shape 赋值为首个输入张量的 Shape。对于 Add 这类输入输出 Shape 一致的算子，无需修改此逻辑；若算子存在输出 Shape 与输入 Shape 不一致的场景，可根据实际需求调整该文件的实现逻辑。可执行以下代码完成文件重写：

In [ ]:
%%writefile Sources/06.03/ops-math/experimental/math/add_custom/op_host/add_custom_infershape.cpp

#include "register/op_impl_registry.h"
#include "log/log.h"

using namespace ge;

namespace ops {
static constexpr int64_t IDX_0 = 0;

static ge::graphStatus InferShapeAddCustom(gert::InferShapeContext* context)
{
    OP_LOGD(context->GetNodeName(), "Begin to do InferShapeAddCustom");

    // get input shapes
    const gert::Shape* xShape = context->GetInputShape(IDX_0);
    OP_CHECK_NULL_WITH_CONTEXT(context, xShape);

    // get output shapes
    gert::Shape* yShape = context->GetOutputShape(IDX_0);
    OP_CHECK_NULL_WITH_CONTEXT(context, yShape);

    // 填充输出shape大小
    auto xShapeSize = xShape->GetDimNum();
    yShape->SetDimNum(xShapeSize);
    for (size_t i = 0; i < xShapeSize; i++) {
        int64_t dim = xShape->GetDim(i);
        yShape->SetDim(i, dim);
    }

    OP_LOGD(context->GetNodeName(), "End to do InferShapeAddCustom");
    return GRAPH_SUCCESS;
}

IMPL_OP_INFERSHAPE(AddCustom).InferShape(InferShapeAddCustom);
} // namespace ops

#### 4.1.2 ${op_name}_graph_infer.cpp
InferDataType函数的作用是根据输入的DataType推导输出的DataType。

**实操示例：**

本次实验中，生成的 infershape 实现文件路径为 experimental/math/add_custom/op_graph/add_custom_graph_infer.cpp，可执行以下命令查看该文件的初始内容：

In [ ]:
!cat ./Sources/06.03/ops-math/experimental/math/add_custom/op_graph/add_custom_graph_infer.cpp

该文件的初始实现逻辑为：直接将算子输出张量的 Dtype 赋值为首个输入张量的 Dtype。对于 Add 这类输入输出 Dtype 一致的算子，无需修改此逻辑；若算子存在输出 Dtype 与输入 Dtype 不一致的场景，可根据实际需求调整该文件的实现逻辑。可执行以下代码完成文件重写：

In [ ]:
%%writefile ./Sources/06.03/ops-math/experimental/math/add_custom/op_graph/add_custom_graph_infer.cpp

#include "register/op_impl_registry.h"
#include "log/log.h"

using namespace ge;
namespace ops {

static ge::graphStatus InferDataTypeForAddCustom(gert::InferDataTypeContext* context)
{
    OP_LOGI("Begin InferDataTypeForAddCustom");
    const ge::DataType x1DataType = context->GetInputDataType(0);
    context->SetOutputDataType(0, x1DataType);
    return ge::GRAPH_SUCCESS;
}

IMPL_OP(AddCustom).InferDataType(InferDataTypeForAddCustom);
} // namespace ops

#### 4.1.3 ${op_name}_proto.h
图模式调用需要将算子原型注册到Graph Engine（简称GE）中，以便GE能够识别该类算子的输入、输出及属性信息。注册通过REG_OP接口完成，开发者需定义算子输入、输出张量类型及数量等基本信息。

常用张量/属性数据类型示例如下：
<div style="overflow-x: auto; float: left; margin: 0 10px 10px 0;">
  <table style="border-collapse: collapse; font-size: 14px; white-space: nowrap;">
    <tr>
      <td align="center"><strong>张量类型</strong></td>
      <td align="left">int64</td>
      <td align="left">int32</td>
      <td align="left">int16</td>
      <td align="left">int8</td>
      <td align="left">double</td>
      <td align="left">float32</td>
      <td align="left">float16</td>
      <td align="left">bfloat16</td>
      <td align="left">complex128</td>
      <td align="left">complex64</td>
      <td align="left">complex32</td>
      <td align="left">/</td>
      <td align="left">/</td>
      <td align="left">/</td>
      <td align="left">/</td>
      <td align="left">/</td>
    </tr>
    <tr>
      <td align="center"><strong>属性类型</strong></td>
      <td align="left">/</td>
      <td align="left">/</td>
      <td align="left">/</td>
      <td align="left">/</td>
      <td align="left">/</td>
      <td align="left">/</td>
      <td align="left">/</td>
      <td align="left">/</td>
      <td align="left">/</td>
      <td align="left">/</td>
      <td align="left">/</td>
      <td align="left">int</td>
      <td align="left">bool</td>
      <td align="left">string</td>
      <td align="left">float</td>
      <td align="left">list</td>
    </tr>
    <tr>
      <td align="center"><strong>示例</strong></td>
      <td align="left">DT_INT64</td>
      <td align="left">DT_INT32</td>
      <td align="left">DT_INT16</td>
      <td align="left">DT_INT8</td>
      <td align="left">DT_DOUBLE</td>
      <td align="left">DT_FLOAT</td>
      <td align="left">DT_FLOAT16</td>
      <td align="left">DT_BF16</td>
      <td align="left">DT_COMPLEX128</td>
      <td align="left">DT_COMPLEX64</td>
      <td align="left">DT_COMPLEX32</td>
      <td align="left">Int</td>
      <td align="left">Bool</td>
      <td align="left">String</td>
      <td align="left">Float</td>
      <td align="left">ListInt</td>
    </tr>
  </table>
</div>
<div style="clear: both;"></div>  
输入输出定义如下：
<table style="border-collapse: collapse; margin: 10px 0; font-size: 14px; width: 100%;">
  <tr style="background-color:#e6f2ff;">
    <th align="center" style="border: 1px solid #ccc; padding: 8px;">输入/输出</th>
    <th align="center" style="border: 1px solid #ccc; padding: 8px;">关键字</th>
    <th align="center" style="border: 1px solid #ccc; padding: 8px;">示例</th>
  </tr>
  <tr>
    <td align="left" style="border: 1px solid #ccc; padding: 8px;">必选输入</td>
    <td align="left" style="border: 1px solid #ccc; padding: 8px;">INPUT</td>
    <td align="left" style="border: 1px solid #ccc; padding: 8px;">.INPUT(<span style="font-size: inherit; font-style: normal;">`${name}`</span>, TensorType({input_dtype}))</td>
  </tr>
  <tr>
    <td align="left" style="border: 1px solid #ccc; padding: 8px;">可选输入</td>
    <td align="left" style="border: 1px solid #ccc; padding: 8px;">OPTIONAL_INPUT</td>
    <td align="left" style="border: 1px solid #ccc; padding: 8px;">.OPTIONAL_INPUT(<span style="font-size: inherit; font-style: normal;">`${name}`</span>, TensorType({optional_input_dtype}))</td>
  </tr>
  <tr>
    <td align="left" style="border: 1px solid #ccc; padding: 8px;">必选属性</td>
    <td align="left" style="border: 1px solid #ccc; padding: 8px;">REQUIRED_ATTR</td>
    <td align="left" style="border: 1px solid #ccc; padding: 8px;">.REQUIRED_ATTR(<span style="font-size: inherit; font-style: normal;">`${name}`</span>, <span style="font-size: inherit; font-style: normal;">`${dtype}`</span>)</td>
  </tr>
  <tr>
    <td align="left" style="border: 1px solid #ccc; padding: 8px;">可选属性</td>
    <td align="left" style="border: 1px solid #ccc; padding: 8px;">ATTR</td>
    <td align="left" style="border: 1px solid #ccc; padding: 8px;">.ATTR(<span style="font-size: inherit; font-style: normal;">`${name}`</span>, <span style="font-size: inherit; font-style: normal;">`${dtype}`</span>, <span style="font-size: inherit; font-style: normal;">`${default_value}`</span>)</td>
  </tr>
  <tr>
    <td align="left" style="border: 1px solid #ccc; padding: 8px;">输出</td>
    <td align="left" style="border: 1px solid #ccc; padding: 8px;">OUTPUT</td>
    <td align="left" style="border: 1px solid #ccc; padding: 8px;">.OUTPUT(<span style="font-size: inherit; font-style: normal;">`${name}`</span>, TensorType({output_dtype}))</td>
  </tr>
</table>
<div style="clear: both;"></div>


**实操示例：**

本次实验中，生成的算子原型注册文件路径为 experimental/math/add_custom/op_graph/add_custom_proto.h，可执行以下命令查看该文件的初始内容：

In [ ]:
!cat ./Sources/06.03/ops-math/experimental/math/add_custom/op_graph/add_custom_proto.h

本算子包含两个输入张量。因此需基于算子原型对相关代码进行重写适配，可执行以下代码完成重写操作。

In [ ]:
%%writefile Sources/06.03/ops-math/experimental/math/add_custom/op_graph/add_custom_proto.h

#ifndef ADD_CUSTOM_PROTO_H_
#define ADD_CUSTOM_PROTO_H_

#include "graph/operator_reg.h"

namespace ge {
REG_OP(AddCustom)
    .INPUT(x1, TensorType({ DT_FLOAT, DT_INT32}))
    .INPUT(x2, TensorType({ DT_FLOAT, DT_INT32}))
    .OUTPUT(y, TensorType({ DT_FLOAT, DT_INT32}))
    .OP_END_FACTORY_REG(AddCustom)
} // namespace ge
#endif // ADD_CUSTOM_PROTO_H_

## 4.2 编译部署
完成所有代码及适配文件的开发后，就可以进行编译部署了

### 4.2.1 算子包编译
编译需要使用开源仓库的build.sh脚本，编译完成后会在build_out目录下生成cann-ops-math-`${vendor_name}`_linux-`${arch}``.run。。由于开发者算子均在experimental目录下，所以常规编译命令如下。
```
bash build.sh --pkg --soc=${soc_version} --vendor_name=${vendor_name} --ops=${op_list} --experimental
```

执行以下命令完成编译：

In [ ]:
%%bash
cd Sources/06.03/ops-math/
bash build.sh --pkg --soc=ascend910b --vendor_name=custom --ops=add_custom --experimental

当提示如下信息，说明编译成功。

```
Self-extractable archive "cann-ops-math-${vendor_name}_linux-${arch}.run" successfully created.
```

编译成功后，run包存放于项目根目录的build_out目录下。

#### 4.2.2 安装自定义算子包
安装自定义算子包的命令如下所示。

```
./build_out/cann-ops-math-${vendor_name}_linux-${arch}.run --insatll-path=XXX
```

其中--install-path为可选参数，用来指定算子的安装路径。本实验中将算子指定安装到`${HOME}`中，最终会安装到`${HOME}`/vendors/customize/op_api下。执行以下命令进行CANN包安装

In [ ]:
%%bash
cd Sources/06.03/ops-math/
./build_out/cann-ops-math-custom_linux-aarch64.run --install-path=${HOME}/

安装完成后，可以执行以下命令查看自动生成的aclnn接口

In [ ]:
!cat ${HOME}/vendors/custom_math/op_api/include/aclnn_add_custom.h

---

## 5.开源仓算子验证
算子验证包含UT验证与算子API调用验证，本节主要介绍算子API调用验证。

开源仓算子与自定义算子的验证流程完全一致：算子安装完成后会对外提供 API 接口，算子验证的核心就是调用这些接口执行算子逻辑。开源仓算子生成后，对应的 API 调用示例代码会自动生成在 examples 目录下，但初始脚本内容为空，需开发者自行编写验证逻辑。

**实操示例：**

本次实验中，生成的算子API调用文件路径为 experimental/math/add_custom/examples/test_aclnn_add_custom.cpp，可执行以下命令查看该文件的初始内容：

In [ ]:
!cat Sources/06.03/ops-math/experimental/math/add_custom/examples/test_aclnn_add_custom.cpp

以下是算子API调用代码，该代码的实现逻辑与[03.03_acl_pybind_call.ipynb](../03_intermediate_vector_operator_development/03.03_acl_pybind_call.ipynb)中介绍的方法完全一致：输入为 shape=[8, 2048] 的float32类型张量，输出同样为 shape=[8, 2048] 的float32类型张量；代码中为两个输入张量分别赋值 1 和 2，调用 add 算子完成加法计算。

In [ ]:
%%writefile Sources/06.03/ops-math/experimental/math/add_custom/examples/test_aclnn_add_custom.cpp
#include <algorithm>
#include <cstdint>
#include <cstdio>
#include <vector>

#include "aclnn/aclnn_base.h"
#include "aclnn/acl_meta.h"
#include "acl/acl_rt.h"
#include "aclnn_add_custom.h"

#define CHECK_ACL(expr)                                                                                 \
    do {                                                                                                \
        auto __ret = (expr);                                                                            \
        int32_t __code = static_cast<int32_t>(__ret);                                                   \
        if (__code != 0) {                                                                              \
            fprintf(stderr, "[ERROR] %s failed at %s:%d, ret=%d\n", #expr, __FILE__, __LINE__, __code); \
        }                                                                                               \
    } while (0)
int32_t main(int32_t argc, char** argv)
{
    const int32_t deviceId = 0;
    aclrtStream stream = nullptr;
    CHECK_ACL(aclnnInit(nullptr));
    CHECK_ACL(aclrtSetDevice(deviceId));
    CHECK_ACL(aclrtCreateStream(&stream));
    const std::vector<int64_t> shape = {8, 2048};
    const int64_t elementCount = shape[0] * shape[1];
    const size_t bufferSize = elementCount * sizeof(float);

    void* input0DeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&input0DeviceMem, bufferSize, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* input0 = aclCreateTensor(shape.data(), shape.size(), ACL_FLOAT, nullptr, 0, ACL_FORMAT_ND,
                                        shape.data(), shape.size(), input0DeviceMem);

    void* input1DeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&input1DeviceMem, bufferSize, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* input1 = aclCreateTensor(shape.data(), shape.size(), ACL_FLOAT, nullptr, 0, ACL_FORMAT_ND,
                                        shape.data(), shape.size(), input1DeviceMem);

    void* output0DeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&output0DeviceMem, bufferSize, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* output0 = aclCreateTensor(shape.data(), shape.size(), ACL_FLOAT, nullptr, 0, ACL_FORMAT_ND,
                                         shape.data(), shape.size(), output0DeviceMem);
    std::vector<float> input0HostData(elementCount, 1.0);
    std::vector<float> input1HostData(elementCount, 2.0);
    std::vector<float> output0HostData(elementCount, 0.0);
    std::vector<float> goldenData(elementCount, 3.0);

    CHECK_ACL(aclrtMemcpy(input0DeviceMem, bufferSize, input0HostData.data(),
                          bufferSize, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL(aclrtMemcpy(input1DeviceMem, bufferSize, input1HostData.data(),
                          bufferSize, ACL_MEMCPY_HOST_TO_DEVICE));
    uint64_t workspaceSize = 0;
    aclOpExecutor* executor = nullptr;
    CHECK_ACL(aclnnAddCustomGetWorkspaceSize(input0, input1, output0, &workspaceSize, &executor));
    void* workspaceDeviceMem = nullptr;
    if (workspaceSize > 0) {
        CHECK_ACL(aclrtMalloc(&workspaceDeviceMem, workspaceSize, ACL_MEM_MALLOC_HUGE_FIRST));
    }
    CHECK_ACL(aclnnAddCustom(workspaceDeviceMem, workspaceSize, executor, stream));
    CHECK_ACL(aclrtSynchronizeStream(stream));
    CHECK_ACL(aclrtMemcpy(output0HostData.data(), bufferSize, output0DeviceMem,
                          bufferSize, ACL_MEMCPY_DEVICE_TO_HOST));

    printf("result is:\n");
    const int64_t previewCount = std::min<int64_t>(elementCount, 10);
    for (int64_t i = 0; i < previewCount; i++) { printf("%.1f ", output0HostData[i]); }
    printf("\ntest %s\n", std::equal(output0HostData.begin(), output0HostData.end(), goldenData.begin()) ? "pass" : "failed");
    aclDestroyTensor(input0);
    aclDestroyTensor(input1);
    aclDestroyTensor(output0);
    CHECK_ACL(aclrtFree(input0DeviceMem));
    CHECK_ACL(aclrtFree(input1DeviceMem));
    CHECK_ACL(aclrtFree(output0DeviceMem));
    if (workspaceSize > 0) {
        CHECK_ACL(aclrtFree(workspaceDeviceMem));
    }
    CHECK_ACL(aclrtDestroyStream(stream));
    CHECK_ACL(aclrtResetDevice(deviceId));
    CHECK_ACL(aclnnFinalize());
    return 0;
}

执行以下代码进行编译运行。

In [ ]:
!cd Sources/06.03/ops-math/;source ${HOME}/vendors/custom_math/bin/set_env.bash;bash build.sh --run_example add_custom eager cust --vendor_name=custom --experimental

调用程序正常执行后会有如下打印：

```
result is:
3.0 3.0 3.0 3.0 3.0 3.0 3.0 3.0 3.0 3.0
test pass

```

---

## 6. 单算子工程迁移
经过前文学习，开发者已基本掌握自定义算子工程与开源仓算子工程的整体结构。本节将详细说明两类工程的核心差异，并分步讲解如何将自定义算子工程迁移为开源仓算子工程，以及最终的提交规范。

### 6.1 工程介绍
基于 Ascend C 开展算子工程化开发时，开发者可选择**自定义算子工程**或**开源算子仓工程**两种形式，两类工程的适用场景与核心优势如下：

<table style="float: left; border-collapse: collapse; margin: 0 10px 10px 0; font-size: 14px;">
  <tr style="background-color:#f0f0f0">
    <td align="center"><strong>工程形式</strong></td>
    <td align="center"><strong>使用场景</strong></td>
    <td align="center"><strong>核心优势</strong></td>
  </tr>
  <tr>
    <td align="left">自定义算子工程</td>
    <td align="left">单个算子的开发、本地快速构建 / 调试 / 功能验证</td>
    <td align="left">工程交付件少，结构轻量化，无需适配多算子编译逻辑，本地开发验证效率高</td>
  </tr>
  <tr>
    <td align="left">开源算子仓工程</td>
    <td align="left">算子贡献、多算子批量构建与出包部署</td>
    <td align="left">原生支持多算子统一编译、打包与部署，交付件按功能模块化拆分，工程结构更规范清晰</td>
  </tr>
</table>
<div style="clear: both;"></div>

### 6.2 交付件差异
以下是两个工程的交付件对比，**自定义算子工程默认使用tiling模板化编程**。

<img src="./images/project_structure_comparison.png" alt="工程结构对比" width="800px">

可以看到核心文件完全一致，核心修改是对文件进行了拆分，接下来再以伪码的形式进行对比，将不同之处进行呈现。

#### 6.2.1 op_host侧修改
<div style="display:flex; width:100%; gap:2%;">
  <!-- 左侧：迁移前 -->
  <div style="width:48%;">
    <p style="font-weight:bold; color:#d32f2f; margin:0 0 4px 0;">自定义算子工程</p>
    <pre style="background:#f5f5f5; padding:10px; border:1px solid #ddd; overflow-x:auto; font-size:11px; margin:0; white-space: pre-wrap; font-family:Consolas, monospace;"><code>// 1. 文件需要拆分，头文件存在差异
#include "register/op_def_registry.h"
#include "../op_kernel/add_tiling.h"
namespace optiling {
const uint32_t BLOCK_DIM = 8;
const uint32_t DEFAULT_TILE_NUM = 8;
constexpr int MIN_LENGTH_FOR_SPLIT = 2048;
static ge::graphStatus TilingFunc(gert::TilingContext *context)
{
    tiling = context->GetTilingData<TilingData>();
    uint32_t totalLength = context->GetInputShape(0)->GetOriginShape().GetShapeSize();
    ge::DataType dtype_x = context->GetInputDesc(0)->GetDataType();
    ...
    tiling->totalLength = totalLength;
    const uint64_t tilingKey = GET_TPL_TILING_KEY(D_T_X, D_T_Y, D_T_Z, TILE_NUM, IS_SPLIT); 
    context->SetTilingKey(tilingKey);
    size_t *currentWorkspace = context->GetWorkspaceSizes(1);
    currentWorkspace[0] = 0;
    return ge::GRAPH_SUCCESS;
}
}
namespace ge {
static graphStatus InferShape(gert::InferShapeContext *context)
{
    ...
}
static graphStatus InferDataType(gert::InferDataTypeContext *context)
{
    ...
}
}
namespace ops {
class Add: public OpDef {
public:
    explicit Add(const char *name) : OpDef(name)
    {
        this->Input("x")
        ...
        this->Output("z")
        ...
        // 2. 需要去掉SetInferShape，开源仓算子需单独调用
        this->SetInferShape(ge::InferShape).SetInferDataType(ge::InferDataType);
        this->AICore()
            // 3. 需要去掉SetTiling，开源仓算子需单独调用
            .SetTiling(optiling::TilingFunc)
            .AddConfig("ascend910b");
    }
};
OP_ADD(Add);
}</code></pre>
  </div>

  <!-- 右侧：迁移后3栏 -->
  <div style="width:48%; display:flex; flex-direction:column; gap:10px;">
    <!-- 右侧第1栏 -->
    <div>
      <p style="font-weight:bold; color:#1976d2; margin:0 0 4px 0;">开源仓算子工程文件1：op_host/add_tiling.cpp</p>
      <pre style="background:#f5f5f5; padding:10px; border:1px solid #ddd; overflow-x:auto; font-size:11px; margin:0; white-space: pre-wrap; font-family:Consolas, monospace;"><code>// 1. 建议使用模板提供的头文件
#include "log/log.h"
#include "util/math_util.h"
#include "op_host/tiling_util.h"
#include "op_host/tiling_templates_registry.h"
#include "../op_kernel/add_tiling_data.h"
#include "../op_kernel/add_tiling_key.h"
namespace optiling {
const uint32_t BLOCK_DIM = 8;
const uint32_t DEFAULT_TILE_NUM = 8;
constexpr int MIN_LENGTH_FOR_SPLIT = 2048;
static ge::graphStatus TilingFunc(gert::TilingContext *context)
{
    tiling = context->GetTilingData<TilingData>();
    uint32_t totalLength = context->GetInputShape(0)->GetOriginShape().GetShapeSize();
    ge::DataType dtype_x = context->GetInputDesc(0)->GetDataType();
    ...
    tiling->totalLength = totalLength;
    const uint64_t tilingKey = GET_TPL_TILING_KEY(D_T_X, D_T_Y, D_T_Z, TILE_NUM, IS_SPLIT); 
    context->SetTilingKey(tilingKey);
    size_t *currentWorkspace = context->GetWorkspaceSizes(1);
    currentWorkspace[0] = 0;
    return ge::GRAPH_SUCCESS;
}
// 2. 该文件中完成TilingFunc注册
IMPL_OP_OPTILING(Add).Tiling(TilingFunc);  
} // namespace optiling</code></pre>
      <p style="font-weight:bold; color:#1976d2; margin:0 0 4px 0;">开源仓算子工程文件2：op_host/add_def.cpp</p>
      <pre style="background:#f5f5f5; padding:10px; border:1px solid #ddd; overflow-x:auto; font-size:11px; margin:0; white-space: pre-wrap; font-family:Consolas, monospace;"><code>// 迁移至op_host/{op_name}_def.cpp后，代码中无SetInferShape和SetTiling内容      
namespace ops {
class Add: public OpDef {
public:
    explicit Add(const char *name) : OpDef(name)
    {
        this->Input("x")
        ...
        this->Output("z")
        ...
        this->AICore()
            .AddConfig("ascend910b");
    }
};
OP_ADD(Add);
}</code></pre>
      <p style="font-weight:bold; color:#1976d2; margin:0 0 4px 0;">开源仓算子工程文件3（可选）：op_host/add_infershape.cpp</p>
      <pre style="background:#f5f5f5; padding:10px; border:1px solid #ddd; overflow-x:auto; font-size:11px; margin:0; white-space: pre-wrap; font-family:Consolas, monospace;"><code>namespace ge {
static graphStatus InferShape(gert::InferShapeContext *context)
{
    ...
}
// 在该文件中完成InferShape注册
IMPL_OP_INFERSHAPE(Add).InferShape(InferShape);
}</code></pre>
    </div>
  </div>
</div>

#### 6.2.2 op_kernel侧修改
<div style="display:flex; width:100%; gap:2%;">
  <!-- 左侧：迁移前 -->
  <div style="width:48%;">
    <p style="font-weight:bold; color:#d32f2f; margin:0 0 4px 0;">自定义算子工程文件：op_kernel/add_tiling.h</p>
    <pre style="background:#f5f5f5; padding:10px; border:1px solid #ddd; overflow-x:auto; font-size:11px; margin:0; white-space: pre-wrap; font-family:Consolas, monospace;"><code>// 文件名不同，其他完全一致
struct AddTilingData {
    int64_t totalNum = 0;
    ...
};</code></pre>
  </div>

  <!-- 右侧：迁移后 -->
  <div style="width:48%; display:flex; flex-direction:column; gap:10px;">
    <div>
      <p style="font-weight:bold; color:#1976d2; margin:0 0 4px 0;">开源仓算子工程文件：op_kernel/add_tiling_data.h</p>
      <pre style="background:#f5f5f5; padding:10px; border:1px solid #ddd; overflow-x:auto; font-size:11px; margin:0; white-space: pre-wrap; font-family:Consolas, monospace;"><code>// 文件名不同，其他完全一致
struct AddTilingData {
    int64_t totalNum = 0;
    ...
};</code></pre>
    </div>
  </div>
</div>

<div style="display:flex; width:100%; gap:2%;">
  <!-- 左侧：迁移前 -->
  <div style="width:48%;">
    <p style="font-weight:bold; color:#d32f2f; margin:0 0 4px 0;">自定义算子工程文件：op_kernel/tiling_key_add.h</p>
    <pre style="background:#f5f5f5; padding:10px; border:1px solid #ddd; overflow-x:auto; font-size:11px; margin:0; white-space: pre-wrap; font-family:Consolas, monospace;"><code>// 文件名不同，其他完全一致
#include "ascendc/host_api/tiling/template_argument.h"
#define ELEMENTWISE_TPL_SCH_MODE_0 0
ASCENDC_TPL_ARGS_DECL(
    Add,
    ASCENDC_TPL_UINT_DECL(schMode, 1, ASCENDC_TPL_UI_LIST, ELEMENTWISE_TPL_SCH_MODE_0));
...
ASCENDC_TPL_SEL(ASCENDC_TPL_ARGS_SEL(
    ASCENDC_TPL_UINT_SEL(schMode, ASCENDC_TPL_UI_LIST, ELEMENTWISE_TPL_SCH_MODE_0)));</code></pre>
  </div>

  <!-- 右侧：迁移后 -->
  <div style="width:48%; display:flex; flex-direction:column; gap:10px;">
    <div>
      <p style="font-weight:bold; color:#1976d2; margin:0 0 4px 0;">开源仓算子工程文件：op_kernel/add_tiling_key.h</p>
      <pre style="background:#f5f5f5; padding:10px; border:1px solid #ddd; overflow-x:auto; font-size:11px; margin:0; white-space: pre-wrap; font-family:Consolas, monospace;"><code>// 文件名不同，其他完全一致
#include "ascendc/host_api/tiling/template_argument.h"
#define ELEMENTWISE_TPL_SCH_MODE_0 0
ASCENDC_TPL_ARGS_DECL(
    Add,
    ASCENDC_TPL_UINT_DECL(schMode, 1, ASCENDC_TPL_UI_LIST, ELEMENTWISE_TPL_SCH_MODE_0));
...
ASCENDC_TPL_SEL(ASCENDC_TPL_ARGS_SEL(
    ASCENDC_TPL_UINT_SEL(schMode, ASCENDC_TPL_UI_LIST, ELEMENTWISE_TPL_SCH_MODE_0)));</code></pre>
    </div>
  </div>
</div>

<div style="display:flex; width:100%; gap:2%;">
  <!-- 左侧：迁移前 -->
  <div style="width:48%;">
    <p style="font-weight:bold; color:#d32f2f; margin:0 0 4px 0;">自定义算子工程文件：op_kernel/add.cpp</p>
    <pre style="background:#f5f5f5; padding:10px; border:1px solid #ddd; overflow-x:auto; font-size:11px; margin:0; white-space: pre-wrap; font-family:Consolas, monospace;"><code>// 文件需要进行拆分，头文件存在差异
#include "kernel_operator.h"
#include "add_tiling.h"
#include "tiling_key_add.h"
#include "kernel_operator_dump_tensor_intf_impl.h"
constexpr int32_t BUFFER_NUM = 1;
template <class dtypeX, class dtypeY, ...>
class KernelAdd {
...
};
template <typename D_T_X, typename D_T_Y, ...>
 __global__ __aicore__ void add(GM_ADDR x, GM_ADDR y, GM_ADDR workspace, GM_ADDR tiling) {
    REGISTER_TILING_DEFAULT(AddTilingData);
    GET_TILING_DATA(tilingData, tiling);
    KernelAdd<D_T_X, D_T_Y,...> op;
    op.Init(x, y ,workspace,  tilingData.totalLength, tilingData.tileNum,tilingData.min);
    if constexpr (std::is_same_v<D_T_X, float> && std::is_same_v<D_T_Y, float>) {
        op.Process();
    } else {
        op.Process2();
    }
}</code></pre>
  </div>

  <!-- 右侧：迁移后2栏 -->
  <div style="width:48%; display:flex; flex-direction:column; gap:10px;">
    <!-- 右侧第1栏 -->
    <div>
      <p style="font-weight:bold; color:#1976d2; margin:0 0 4px 0;">开源仓算子工程文件1：op_kernel/add.h</p>
      <pre style="background:#f5f5f5; padding:10px; border:1px solid #ddd; overflow-x:auto; font-size:11px; margin:0; white-space: pre-wrap; font-family:Consolas, monospace;"><code>// 导入的头文件存在部分差异
#include "kernel_operator.h"
#include "kernel_tiling/kernel_tiling.h"
#include "add_tiling_data.h"
#include "add_tiling_key.h"
constexpr int32_t BUFFER_NUM = 1;
template <class dtypeX, class dtypeY, ...>
class KernelAdd {
...
};</code></pre>
      <p style="font-weight:bold; color:#1976d2; margin:0 0 4px 0;">开源仓算子工程文件2：op_kernel/add.cpp</p>
      <pre style="background:#f5f5f5; padding:10px; border:1px solid #ddd; overflow-x:auto; font-size:11px; margin:0; white-space: pre-wrap; font-family:Consolas, monospace;"><code>// 导入的头文件存在部分差异   
#include "add.h"   
template <typename D_T_X, typename D_T_Y, ...>
 __global__ __aicore__ void add(GM_ADDR x, GM_ADDR y, GM_ADDR workspace, GM_ADDR tiling) {
    REGISTER_TILING_DEFAULT(AddTilingData);
    GET_TILING_DATA(tilingData, tiling);
    KernelAdd<D_T_X, D_T_Y,...> op;
    op.Init(x, y ,workspace,  tilingData.totalLength, tilingData.tileNum,tilingData.min);
    if constexpr (std::is_same_v<D_T_X, float> && std::is_same_v<D_T_Y, float>) {
        op.Process();
    } else {
        op.Process2();
    }
}</code></pre>
    </div>
  </div>
</div>

### 6.3 总结
从自定义算子开发到完成开源仓算子贡献的全流程来看，整体迁移适配路径清晰、操作便捷，核心需重点关注以下关键环节：

1. 开源仓迁移核心注意事项
   - **工程模板复用**：基于开源仓 genop 工具生成标准化算子工程，直接复用模板内置的头文件（如 tiling_util.h、tiling_templates_registry.h），无需手动适配头文件路径，降低适配成本；
   - **文件拆分规范**：需将原自定义算子工程中耦合的 tiling 逻辑与 kernel 逻辑拆分至独立文件（如 add_tiling.cpp、add_kernel.cpp），遵循开源仓文件目录规范；
   - **函数调用适配**：拆分后 tiling、InferShape 等核心函数的注册 / 调用方式需调整（如 IMPL_OP_OPTILING(Add).Tiling(TilingFunc) 替代原 SetTiling 调用），需同步更新函数绑定逻辑。

2. 开源贡献补充工作
   完成上述迁移后，需补充以下开源合规性开发工作，确保算子满足仓内接入标准：
   - **单算子 API 封装**：编写标准化的单算子 API 调用示例代码，覆盖入参校验、算子实例化、执行流程等核心场景，便于仓内集成验证；
   - **单元测试（UT）开发**：编写覆盖算子核心逻辑的 UT 用例（下一节将详细介绍 UT 编写规范与示例），验证算子功能正确性、边界条件兼容性。

完成上述所有开发工作后，需在本地环境完成算子工程的编译构建，并通过 UT 用例全量验证，确认算子功能、性能符合预期，即可完成整算子开发与开源贡献的全流程。

---

## 课后实践
请根据本节教程内容，在下面的代码框中补充实现sub算子，使用Ascend C编程实现矢量减法算子sub_custom。

- 数据类型：输入与输出均为 float 类型。
- 数据形状（Shape）： 输入Shape为 (8, 2048)，输出Shape与输入相同。
- 数据布局（Format）：ND。

**创建算子工程**

In [ ]:
%%bash
cd Sources/06.03/ops-math/
# 预防重复生成，先删除sub_custom目录
rm -rf experimental/math/sub_custom
# 创建新的算子工程
bash build.sh --genop=experimental/math/sub_custom

**确认原型定义**

这里已给出一份原型定义，输入为x1、x2，输出为y，支持的数据类型为float，请执行覆盖工程生成的定义文件。

In [ ]:
%%writefile  Sources/06.03/ops-math/experimental/math/sub_custom/op_host/sub_custom_def.cpp
#include "register/op_def_registry.h"

namespace ops {
class SubCustom : public OpDef {
public:
    explicit SubCustom(const char* name) : OpDef(name)
    {
        // 输入参数说明
        this->Input("x1") 
            .ParamType(REQUIRED) 
            .DataType({ge::DT_FLOAT}) 
            .Format({ge::FORMAT_ND}) 
            .UnknownShapeFormat({ge::FORMAT_ND})
            .AutoContiguous(); 
        this->Input("x2") 
            .ParamType(REQUIRED) 
            .DataType({ge::DT_FLOAT}) 
            .Format({ge::FORMAT_ND}) 
            .UnknownShapeFormat({ge::FORMAT_ND})
            .AutoContiguous();
        this->Output("y") 
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT})
            .Format({ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND})
            .AutoContiguous();

        this->AICore().AddConfig("ascend910b");
    }
};
OP_ADD(SubCustom); // 添加算子信息库
} // namespace ops


**完成tiling结构体定义**

请根据需求修改Tiling结构体

In [ ]:
%%writefile  Sources/06.03/ops-math/experimental/math/sub_custom/op_kernel/sub_custom_tiling_data.h

#ifndef __SUB_CUSTOM_TILLING_DATA_H__
#define __SUB_CUSTOM_TILLING_DATA_H__

struct SubCustomTilingData {
    int64_t totalLength;
    int64_t tileNum;
    // 扩展其他tilling参数
};
#endif

**添加模板定义**

这里已给出一份基础的模板定义，请根据需要修改。

In [ ]:
%%writefile Sources/06.03/ops-math/experimental/math/sub_custom/op_kernel/sub_custom_tiling_key.h


#ifndef __SUB_CUSTOM_TILING_KEY_H__
#define __SUB_CUSTOM_TILING_KEY_H__

#include "ascendc/host_api/tiling/template_argument.h"

/* Mode场景定义 */
#define ELEMENTWISE_TPL_SCH_MODE_0 0
#define ELEMENTWISE_TPL_SCH_MODE_1 1
/* 继续定义其他Mode场景... */

/* 模板参数 */
ASCENDC_TPL_ARGS_DECL(
    SubCustom,
    ASCENDC_TPL_UINT_DECL(schMode, 1, ASCENDC_TPL_UI_LIST, ELEMENTWISE_TPL_SCH_MODE_0, ELEMENTWISE_TPL_SCH_MODE_1));

/* 模板参数组合 */
ASCENDC_TPL_SEL(ASCENDC_TPL_ARGS_SEL(
    ASCENDC_TPL_UINT_SEL(schMode, ASCENDC_TPL_UI_LIST, ELEMENTWISE_TPL_SCH_MODE_0, ELEMENTWISE_TPL_SCH_MODE_1)));

#endif


**完成tiling函数**

请根据需求修改Tiling函数

In [ ]:
%%writefile  Sources/06.03/ops-math/experimental/math/sub_custom/op_host/sub_custom_tiling.cpp

#include "log/log.h"
#include "util/math_util.h"
#include "op_host/tiling_util.h"
#include "op_host/tiling_templates_registry.h"
#include "../op_kernel/sub_custom_tiling_data.h"
#include "../op_kernel/sub_custom_tiling_key.h"

namespace optiling {

struct SubCustomCompileInfo {};

// tiling 分发入口
static ge::graphStatus SubCustomTilingFunc(gert::TilingContext* context)
{
    return ge::GRAPH_SUCCESS;
}

static ge::graphStatus TilingParseForSubCustom([[maybe_unused]] gert::TilingParseContext* context)
{   
    // AscendC 算子可以直接返回SUCCESS提升性能，硬件信息可在运行时获取
    return ge::GRAPH_SUCCESS;
}

// tiling注册入口.
IMPL_OP_OPTILING(SubCustom).Tiling(SubCustomTilingFunc).TilingParse<SubCustomCompileInfo>(TilingParseForSubCustom);
} // namespace optiling

**完成infershape函数（可选）**

以下为模板初始化的infershape函数，请根据需要进行修改

In [ ]:
%%writefile Sources/06.03/ops-math/experimental/math/sub_custom/op_host/sub_custom_infershape.cpp

#include "register/op_impl_registry.h"
#include "log/log.h"

using namespace ge;

namespace ops {
static constexpr int64_t IDX_0 = 0;
static ge::graphStatus InferShapeSubCustom(gert::InferShapeContext* context)
{
    OP_LOGD(context->GetNodeName(), "Begin to do InferShapeSubCustom");

    // get input shapes
    const gert::Shape* xShape = context->GetInputShape(IDX_0);
    OP_CHECK_NULL_WITH_CONTEXT(context, xShape);

    // get output shapes
    gert::Shape* yShape = context->GetOutputShape(IDX_0);
    OP_CHECK_NULL_WITH_CONTEXT(context, yShape);

    // 填充输出shape大小
    auto xShapeSize = xShape->GetDimNum();
    yShape->SetDimNum(xShapeSize);
    for (size_t i = 0; i < xShapeSize; i++) {
        int64_t dim = xShape->GetDim(i);
        yShape->SetDim(i, dim);
    }

    OP_LOGD(context->GetNodeName(), "End to do InferShapeSubCustom");
    return GRAPH_SUCCESS;
}

IMPL_OP_INFERSHAPE(SubCustom).InferShape(InferShapeSubCustom);
} // namespace ops

**完成核函数开发**

请根据需求修改核函数

In [ ]:
%%writefile Sources/06.03/ops-math/experimental/math/sub_custom/op_kernel/sub_custom.cpp

#include "sub_custom.h"

enum class SubTilingKey : uint32_t
{
    TILING_KEY_EXAMPLE_FLOAT = 0,
    TILING_KEY_EXAMPLE_INT32 = 1,
};

template <uint32_t schMode>
__global__ __aicore__ void sub_custom(GM_ADDR x, GM_ADDR y, GM_ADDR z, GM_ADDR workspace, GM_ADDR tiling)
{
    REGISTER_TILING_DEFAULT(SubCustomTilingData);
    GET_TILING_DATA_WITH_STRUCT(SubCustomTilingData, tilingData, tiling);

    // 场景1
    if constexpr (schMode == static_cast<uint32_t>(SubTilingKey::TILING_KEY_EXAMPLE_FLOAT)) {
        NsSub::Sub<float> op; // 算子kernel实例获取
        op.Init(x, y, z, &tilingData);      // 算子kernel实例初始化
        op.Process();                       // 算子kernel实例执行
    }

    // 场景2
    if constexpr (schMode == static_cast<uint32_t>(SubTilingKey::TILING_KEY_EXAMPLE_INT32)) {
        NsSub::Sub<int32_t> op; // 算子kernel实例获取
        op.Init(x, y, z, &tilingData);        // 算子kernel实例初始化
        op.Process();                         // 算子kernel实例执行
    }
}

**完成算法实现**

请根据需求修改算法实现

In [ ]:
%%writefile Sources/06.03/ops-math/experimental/math/sub_custom/op_kernel/sub_custom.h

#ifndef __SUB_CUSTOM_H__
#define __SUB_CUSTOM_H__

#include "kernel_operator.h"
#include "kernel_tiling/kernel_tiling.h"
#include "sub_custom_tiling_data.h"
#include "sub_custom_tiling_key.h"

namespace NsSub {

using namespace AscendC;

constexpr int32_t BUFFER_NUM = 2;

template <typename T>
class Sub {
public:
    __aicore__ inline Sub(){};

    __aicore__ inline void Init(/*参数列表*/);
    __aicore__ inline void Process(/*参数列表*/);

private:
    __aicore__ inline void CopyIn(/*参数列表*/);
    __aicore__ inline void CopyOut(/*参数列表*/);
    __aicore__ inline void Compute(/*参数列表*/);

private:
    TPipe pipe;
    TQue<QuePosition::VECIN, BUFFER_NUM> XXX;
    TQue<QuePosition::VECOUT, BUFFER_NUM> YYY;
};

template <typename T>
__aicore__ inline void Sub<T>::Init(/*参数列表*/)
{
}

template <typename T>
__aicore__ inline void Sub<T>::CopyIn(/*参数列表*/)
{
}

template <typename T>
__aicore__ inline void Sub<T>::CopyOut(/*参数列表*/)
{
}

template <typename T>
__aicore__ inline void Sub<T>::Compute(/*参数列表*/)
{
}

template <typename T>
__aicore__ inline void Sub<T>::Process()
{
    for (int32_t i = 0; i < /*循环次数*/; i++) {
        CopyIn(/*参数列表*/);
        Compute(/*参数列表*/);
        CopyOut(/*参数列表*/);
    }
}

} // namespace NsSub
#endif // SUB_CUSTOM_H

**完成算子API调用代码**

请根据需求完成算子API调用代码

In [ ]:
%%writefile Sources/06.03/ops-math/experimental/math/sub_custom/examples/test_aclnn_sub_custom.cpp
#include <algorithm>
#include <cstdint>
#include <cstdio>
#include <vector>

#include "aclnn/aclnn_base.h"
#include "aclnn/acl_meta.h"
#include "acl/acl_rt.h"
#include "aclnn_sub_custom.h"

#define CHECK_ACL(expr)                                                                                 \
    do {                                                                                                \
        auto __ret = (expr);                                                                            \
        int32_t __code = static_cast<int32_t>(__ret);                                                   \
        if (__code != 0) {                                                                              \
            fprintf(stderr, "[ERROR] %s failed at %s:%d, ret=%d\n", #expr, __FILE__, __LINE__, __code); \
        }                                                                                               \
    } while (0)
int32_t main(int32_t argc, char** argv)
{
    const int32_t deviceId = 0;
    aclrtStream stream = nullptr;
    CHECK_ACL(aclnnInit(nullptr));
    CHECK_ACL(aclrtSetDevice(deviceId));
    CHECK_ACL(aclrtCreateStream(&stream));
    const std::vector<int64_t> shape = {8, 2048};
    const int64_t elementCount = shape[0] * shape[1];
    const size_t bufferSize = elementCount * sizeof(float);

    void* input0DeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&input0DeviceMem, bufferSize, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* input0 = aclCreateTensor(shape.data(), shape.size(), ACL_FLOAT, nullptr, 0, ACL_FORMAT_ND,
                                        shape.data(), shape.size(), input0DeviceMem);

    void* input1DeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&input1DeviceMem, bufferSize, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* input1 = aclCreateTensor(shape.data(), shape.size(), ACL_FLOAT, nullptr, 0, ACL_FORMAT_ND,
                                        shape.data(), shape.size(), input1DeviceMem);

    void* output0DeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&output0DeviceMem, bufferSize, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* output0 = aclCreateTensor(shape.data(), shape.size(), ACL_FLOAT, nullptr, 0, ACL_FORMAT_ND,
                                         shape.data(), shape.size(), output0DeviceMem);
    std::vector<float> input0HostData(elementCount, 1.0);
    std::vector<float> input1HostData(elementCount, 2.0);
    std::vector<float> output0HostData(elementCount, 0.0);
    std::vector<float> goldenData(elementCount, -1.0);

    CHECK_ACL(aclrtMemcpy(input0DeviceMem, bufferSize, input0HostData.data(),
                          bufferSize, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL(aclrtMemcpy(input1DeviceMem, bufferSize, input1HostData.data(),
                          bufferSize, ACL_MEMCPY_HOST_TO_DEVICE));
    uint64_t workspaceSize = 0;
    aclOpExecutor* executor = nullptr;
    CHECK_ACL(aclnnSubCustomGetWorkspaceSize(input0, input1, output0, &workspaceSize, &executor));
    void* workspaceDeviceMem = nullptr;
    if (workspaceSize > 0) {
        CHECK_ACL(aclrtMalloc(&workspaceDeviceMem, workspaceSize, ACL_MEM_MALLOC_HUGE_FIRST));
    }
    CHECK_ACL(aclnnSubCustom(workspaceDeviceMem, workspaceSize, executor, stream));
    CHECK_ACL(aclrtSynchronizeStream(stream));
    CHECK_ACL(aclrtMemcpy(output0HostData.data(), bufferSize, output0DeviceMem,
                          bufferSize, ACL_MEMCPY_DEVICE_TO_HOST));

    printf("result is:\n");
    const int64_t previewCount = std::min<int64_t>(elementCount, 10);
    for (int64_t i = 0; i < previewCount; i++) { printf("%.1f ", output0HostData[i]); }
    printf("\ntest %s\n", std::equal(output0HostData.begin(), output0HostData.end(), goldenData.begin()) ? "pass" : "failed");
    aclDestroyTensor(input0);
    aclDestroyTensor(input1);
    aclDestroyTensor(output0);
    CHECK_ACL(aclrtFree(input0DeviceMem));
    CHECK_ACL(aclrtFree(input1DeviceMem));
    CHECK_ACL(aclrtFree(output0DeviceMem));
    if (workspaceSize > 0) {
        CHECK_ACL(aclrtFree(workspaceDeviceMem));
    }
    CHECK_ACL(aclrtDestroyStream(stream));
    CHECK_ACL(aclrtResetDevice(deviceId));
    CHECK_ACL(aclnnFinalize());
    return 0;
}

**运行测试**

In [ ]:
%%bash
cd Sources/06.03/ops-math/
# 编译算子工程并部署编译出的算子包
bash build.sh --pkg --soc=ascend910b --vendor_name=custom --ops=sub_custom --experimental;./build_out/cann-ops*.run --install-path=${HOME}/

# 部署算子后编译调用代码
source ${HOME}/vendors/custom_math/bin/set_env.bash;bash build.sh --run_example sub_custom eager cust --vendor_name=custom --experimental

**查看答案**

实现方式不唯一，这里给出的答案仅供参考。  

执行命令查看Tiling结构体参考答案：

In [ ]:
!cat ./answer/06.03_answer/op_kernel/sub_custom_tiling_data.h

执行命令查看Tiling函数参考答案：

In [ ]:
!cat ./answer/06.03_answer/op_host/sub_custom_tiling.cpp

执行命令查看infershape函数参考答案：

In [ ]:
!cat ./answer/06.03_answer/op_host/sub_custom_infershape.cpp

执行命令查看核函数参考答案：

In [ ]:
!cat ./answer/06.03_answer/op_kernel/sub_custom.cpp

执行命令查看算法实现参考答案：

In [ ]:
!cat ./answer/06.03_answer/op_kernel/sub_custom.h

执行命令查看测试代码参考答案:

In [ ]:
!cat ./answer/06.03_answer/examples/test_aclnn_sub_custom.cpp